# Detección de neumonía en radiografías de tórax
Comparación de una CNN entrenada desde cero contra transfer learning (MobileNetV2).

Dataset: Chest X-Ray Images (Pneumonia), Kermany et al. (2018).

In [ ]:
import os
import time
import glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, ConfusionMatrixDisplay

tf.keras.utils.set_random_seed(42)
os.makedirs("figuras", exist_ok=True)

In [ ]:
ruta = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
ruta = os.path.join(ruta, "chest_xray")
clases = ["NORMAL", "PNEUMONIA"]

def listar(particion):
    rutas, etiquetas = [], []
    for i, c in enumerate(clases):
        archivos = sorted(glob.glob(os.path.join(ruta, particion, c, "*.jpeg")))
        rutas += archivos
        etiquetas += [i] * len(archivos)
    return np.array(rutas), np.array(etiquetas)

for particion in ["train", "val", "test"]:
    r, e = listar(particion)
    print(f"{particion:5s}: {len(r):5d} imagenes  NORMAL = {(e == 0).sum():4d}  PNEUMONIA = {(e == 1).sum():4d}")

## División de datos
La carpeta `val` original solo tiene 16 imágenes, así que se juntan `train` y `val` y se separa un 15 % estratificado para validación. La carpeta `test` se deja intacta para la evaluación final.

In [ ]:
rutas_tr, y_tr = listar("train")
rutas_va, y_va = listar("val")
rutas_te, y_te = listar("test")

rutas_todas = np.concatenate([rutas_tr, rutas_va])
y_todas = np.concatenate([y_tr, y_va])
rutas_train, rutas_val, y_train, y_val = train_test_split(rutas_todas, y_todas, test_size=0.15, stratify=y_todas, random_state=42)
rutas_test, y_test = rutas_te, y_te

for nombre, y in [("entrenamiento", y_train), ("validacion", y_val), ("prueba", y_test)]:
    print(f"{nombre:13s}: {len(y):5d}  NORMAL = {(y == 0).sum():4d} ({(y == 0).mean() * 100:.1f} %)  PNEUMONIA = {(y == 1).sum():4d} ({(y == 1).mean() * 100:.1f} %)")

In [ ]:
IMG = 160
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

def cargar(ruta_img, etiqueta):
    # escala de grises redimensionada y guardada como uint8 para que quepa en memoria
    img = tf.io.decode_jpeg(tf.io.read_file(ruta_img), channels=1)
    img = tf.image.resize(img, (IMG, IMG))
    return tf.cast(img, tf.uint8), etiqueta

def a_rgb(img, etiqueta):
    # mobilenetv2 espera 3 canales
    return tf.image.grayscale_to_rgb(tf.cast(img, tf.float32)), etiqueta

def crear_dataset(rutas, y, mezclar):
    ds = tf.data.Dataset.from_tensor_slices((rutas, y.astype("float32")))
    ds = ds.map(cargar, num_parallel_calls=AUTOTUNE).cache()
    if mezclar:
        ds = ds.shuffle(len(rutas), seed=42)
    return ds.map(a_rgb, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

train_ds = crear_dataset(rutas_train, y_train, True)
val_ds = crear_dataset(rutas_val, y_val, False)
test_ds = crear_dataset(rutas_test, y_test, False)

In [ ]:
fig, ejes = plt.subplots(2, 4, figsize=(12, 6))
for fila, c in enumerate(clases):
    ejemplos = rutas_train[y_train == fila][:4]
    for k, r in enumerate(ejemplos):
        img, _ = cargar(r, 0)
        ejes[fila, k].imshow(img.numpy().squeeze(), cmap="gray")
        ejes[fila, k].set_title(c)
        ejes[fila, k].axis("off")
plt.tight_layout()
plt.savefig("figuras/ejemplos.png", dpi=150)
plt.show()

In [ ]:
# pesos por clase para compensar el desbalance
n = len(y_train)
pesos_clase = {0: n / (2 * (y_train == 0).sum()), 1: n / (2 * (y_train == 1).sum())}
print("pesos por clase:", {clases[k]: round(float(v), 3) for k, v in pesos_clase.items()})